# Setup and Configuration

In [ ]:
# Parameters for FRED API calls
fred_api_key = "[Paste Federal Reserve API Key]"
fred_release = "[Paste Federal Reserve Release ID (integer)]"

# Parameter for Date Table date range
years_back: int = 5

StatementMeta(, f496444b-e40c-493c-ad08-4a5800396a85, 3, Finished, Available, Finished)

In [2]:
from datetime import date, datetime
import requests
from pyspark.sql import functions as F

StatementMeta(, f496444b-e40c-493c-ad08-4a5800396a85, 4, Finished, Available, Finished)

In [3]:
# Parameters for FRED API calls
fred_url = "https://api.stlouisfed.org/fred/v2/release/observations"
fred_params = {
    "release_id" : fred_release,
    "format" : "json"
}
fred_headers = {
    "Authorization" : f"Bearer {fred_api_key}"
}

StatementMeta(, f496444b-e40c-493c-ad08-4a5800396a85, 5, Finished, Available, Finished)

In [4]:
# Variables for FRED filtering and Date table
current_datetime = datetime.now()
current_year = current_datetime.year
start_year = current_year - years_back

# Date Range
start_date = date(start_year, 1, 1)
end_date = date(current_year + 1, 12, 31)

StatementMeta(, f496444b-e40c-493c-ad08-4a5800396a85, 6, Finished, Available, Finished)

# Date Table

In [5]:
# 1. Generate the sequence of dates
df_dates = spark.sql(f"SELECT explode(sequence(to_date('{start_date}'), to_date('{end_date}'), interval 1 day)) as date")

# 2. Add dimension columns
dim_date = df_dates.select(
    F.col("date"),
    F.year("date").alias("year"),
    F.month("date").alias("month_number"),
    F.date_format("date", "MMMM").alias("month_name"),
    F.date_format("date", "MMM").alias("month_name_short"),
    F.quarter("date").alias("quarter"),
    F.date_format("date", "EEEE").alias("day_name"),
    F.dayofweek("date").alias("day_of_week_number"),
    # Create a YearMonth key for sorting (e.g., 202401)
    (F.year("date") * 100 + F.month("date")).alias("year_month_key")
).withColumn(
    "MMM-YYYY", F.concat_ws("-", F.col("month_name_short"), F.col("year"))
).withColumn(
        "week_start_date", F.date_trunc("week", F.col("date")).cast("date")
).withColumn(
    "week_end_date", F.date_add("week_start_date",6)
)



StatementMeta(, f496444b-e40c-493c-ad08-4a5800396a85, 7, Finished, Available, Finished)

# FRED API Call(s)

In [6]:
# Extracts data from FRED API and appends to a list
data = []

has_more = True

while has_more == True:
    response = requests.get(
        url= fred_url,
        headers= fred_headers,
        params=fred_params )
    if response.status_code == 200:
        data.append(response.json())
    has_more = response.json()["has_more"]
    if has_more == True:
        fred_params["next_cursor"] = response.json()["next_cursor"]


StatementMeta(, f496444b-e40c-493c-ad08-4a5800396a85, 8, Finished, Available, Finished)

# FRED data to Spark Dataframe

In [7]:
# Define of FRED release observations
ddl_schema = """
    has_more BOOLEAN,
    release STRUCT<release_id: INT, name: STRING, url: STRING, sources: ARRAY<STRUCT<
        name: STRING, url: STRING>> >,
    series ARRAY<STRUCT<
        series_id: STRING, 
        title: STRING,
        frequency: STRING,
        units: STRING,
        seasonal_adjustment: STRING,
        last_update: STRING,
        copyright_id: STRING,
        notes: STRING,
        observations: ARRAY<STRUCT<date: STRING, value: STRING>>
    >>
"""

# Apply it during creation
df = spark.createDataFrame(data= data, schema=ddl_schema)

StatementMeta(, f496444b-e40c-493c-ad08-4a5800396a85, 9, Finished, Available, Finished)

In [8]:
# Extracts and structures observation data
df_observations = (
    df.select(
        F.explode("series").alias("s")
    )
    .select("s.*")
    .select(
        "series_id",
        F.explode("observations").alias("obs")
        )
    .select(
        "series_id",
        F.col("obs.date").cast("date").alias("date"),
        F.col("obs.value").cast("double").alias("value")
    )
    .join(df_dates, "date", "left_semi")
)

StatementMeta(, f496444b-e40c-493c-ad08-4a5800396a85, 10, Finished, Available, Finished)

In [9]:
# Extracts and structures release and series data
df_series = (
    df.select(
        "release.*",
        F.explode("series").alias("s")
    )
    .select("s.*")
    .drop("observations")
    .drop_duplicates()
    .join(df_observations, "series_id", "left_semi")
    .withColumn("updated", F.lit(current_datetime))
)

StatementMeta(, f496444b-e40c-493c-ad08-4a5800396a85, 11, Finished, Available, Finished)

# Write FRED data to lakehouse

In [10]:
# Write date table to dbo schema
dim_date.write.format("delta").mode("overwrite").saveAsTable("dbo.dim_date")

StatementMeta(, f496444b-e40c-493c-ad08-4a5800396a85, 12, Finished, Available, Finished)

In [11]:
# Writes series data to dbo schema
(
    df_series.write.format("delta")
    .mode("overwrite")
    .saveAsTable("dbo.dim_fred_series")
)

StatementMeta(, f496444b-e40c-493c-ad08-4a5800396a85, 13, Finished, Available, Finished)

In [12]:
# Writes observation data to dbo schema
(
    df_observations.write.format("delta")
    .mode("overwrite")
    .saveAsTable("dbo.fact_fred_observations")
)

StatementMeta(, f496444b-e40c-493c-ad08-4a5800396a85, 14, Finished, Available, Finished)